# 3.4 Linear Regression Implementation from Scratch (PyTorch)

We are now ready to work through a fully functioning implementation of linear regression. In this section, we will implement the entire method from scratch, including:
1.  The model
2.  The loss function
3.  A minibatch stochastic gradient descent optimizer
4.  The training function that stitches all of these pieces together

Finally, we will run our synthetic data generator and apply our model on the resulting dataset.

In [ ]:
%matplotlib inline
import torch
import random
from torch.utils import data
import matplotlib.pyplot as plt

## 3.4.1. Generating the Dataset
To simulate a real-world scenario, we generate a synthetic dataset where we know the true parameters. We construct a dataset according to the linear model:

$$\mathbf{y} = \mathbf{X} \mathbf{w} + b + \epsilon$$

Where $\epsilon$ is random noise.
* **True Weights ($\mathbf{w}$):** `[2, -3.4]`
* **True Bias ($b$):** `4.2`
* **Features:** 1000 examples with 2 features each.

In [ ]:
def synthetic_data(w,b,num_examples):
    X=torch.normal(0,1,(num_examples,len(w)))
    y=torch.matmul(X,w)+b
    y+=torch.normal(0,0.01,y.shape)
    return X,y.reshape((-1,1))
true_w=torch.tensor([2,-3.4])
true_b=4.2
features,labels=synthetic_data(true_w,true_b,1000)
print('features:', features[0])
print('label:', labels[0])

## 3.4.2. Reading the Dataset
Training models requires iterating over the dataset and grabbing a small batch of data (a **minibatch**) at a time. This is essential for Stochastic Gradient Descent (SGD).

Instead of custom loaders, we use PyTorch's standard `TensorDataset` and `DataLoader`. These utilities handle shuffling and batching efficiently.

In [ ]:
def load_array(data_arrays,batch_size,is_train=True):
    dataset=data.TensorDataset(*data_arrays)
    return data.Dataloader(dataset,batch_size,shuffle=is_train)
batch_size=10
data_iter=load_array()